In [ ]:
from pathlib import Path

from pdfminer.high_level import extract_text as pdfminer_extract_text
from unstructured.chunking.title import chunk_by_title
from unstructured.cleaners.core import clean_extra_whitespace, clean_non_ascii_chars
from unstructured.documents.elements import Element
from unstructured.partition.auto import partition
from unstructured.partition.pdf import partition_pdf
from unstructured.partition.text import partition_text
from unstructured.documents.elements import (
    Element,
    Header,
    Footer,
)
from sentence_transformers import SentenceTransformer

faq_urls = [
    "https://teaspoonofadventure.com/75-questions-for-travellers/",
    "https://www.adventure-life.com/rwanda/articles/rwanda-faqs",
]
print("Starting script...")
print(f"FAQ URLs: {faq_urls}")

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data"

pdf_paths = [
    str(DATA_DIR / "TravelTips-Oct2008.PDF"),
    str(DATA_DIR / "The-Best-100-Travel-Tips-and-Hacks-by-Jessica-Ufuoma-1.pdf"),
]
print(f"PDF Paths: {pdf_paths}")


# Partition articles urls
def partition_article(urls):
    results = {}

    for url in urls:
        results[url] = partition(url=url)

    return results


# Partition pdf documents
def partition_pdf_documents(pdf_paths):
    results = {}

    for pdf_path in pdf_paths:
        elements = partition_pdf(filename=pdf_path, strategy="fast")

        if not elements:
            # unstructured's complexity heuristic can flag an entire document as
            # "too complex" for text extraction based on a single dense-graphics
            # page, skipping extraction even under the fast strategy. Fall back to
            # pdfminer's raw text extraction in that case.
            text = pdfminer_extract_text(pdf_path)
            elements = partition_text(text=text)

        results[pdf_path] = elements

    return results


# Clean the elements by removing empty or irrelevant text
def clean_elements(elements):
    cleaned = []

    for element in elements:
        if not element.text:
            continue

        text = clean_extra_whitespace(element.text)
        text = clean_non_ascii_chars(text)

        if not text.strip():
            continue

        element.text = text
        cleaned.append(element)

    return cleaned


# Filter unecessary elements from the documents and articles
def filter_elements(elements):
    filtered = []

    for element in elements:
        if not element.text:
            continue

        text = element.text.strip()

        if not text:
            continue

        # Remove very short fragments
        if len(text) < 30:
            continue

        # Remove headers and footers
        if isinstance(element, (Header, Footer)):
            continue

        filtered.append(element)

    return filtered


def preprocess_elements(elements):
    elements = clean_elements(elements)
    elements = filter_elements(elements)

    return elements


# save metadata for each source
def save_metadata(elements, source):
    for element in elements:
        element.metadata.source = source
    return elements


def preprocess_and_chunk(elements, source):
    elements = save_metadata(elements, source)
    elements = clean_elements(elements)
    elements = filter_elements(elements)
    elements = chunk_by_title(
        elements,
        max_characters=4000,
        new_after_n_chars=3000,
        combine_text_under_n_chars=500,
    )

    return elements


print("Starting script...")

articles = partition_article(faq_urls)
print("Finished partitioning articles")

documents = partition_pdf_documents(pdf_paths)
print("Finished partitioning PDFs")

chunked_articles = {
    source: preprocess_and_chunk(elements, source)
    for source, elements in articles.items()
}

chunked_documents = {
    source: preprocess_and_chunk(elements, source)
    for source, elements in documents.items()
}


# Check the outputs for articles
for source, chunks in chunked_articles.items():
    print(f"\n{'=' * 80}")
    print(f"SOURCE: {source}")
    print(f"CHUNKS: {len(chunks)}")
    print(f"{'=' * 80}")

    for i, chunk in enumerate(chunks):
        print(f"\n--- Chunk {i} ---")
        print(chunk.text)
        print(f"Characters: {len(chunk.text)}")
        print(f"Metadata: {chunk.metadata.to_dict()}")

# Check the outputs for pdf documents
for source, chunks in chunked_documents.items():
    print(f"\n{'=' * 80}")
    print(f"SOURCE: {source}")
    print(f"CHUNKS: {len(chunks)}")
    print(f"{'=' * 80}")

    for i, chunk in enumerate(chunks):
        print(f"\n--- Chunk {i} ---")
        print(chunk.text)
        print(f"Characters: {len(chunk.text)}")
        print(f"Metadata: {chunk.metadata.to_dict()}")

c:\Users\Johnson\Desktop\vacation_planner\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting script...
FAQ URLs: ['https://teaspoonofadventure.com/75-questions-for-travellers/', 'https://www.adventure-life.com/rwanda/articles/rwanda-faqs']
PDF Paths: ['c:\\Users\\Johnson\\Desktop\\vacation_planner\\src\\itineraries\\data\\TravelTips-Oct2008.PDF', 'c:\\Users\\Johnson\\Desktop\\vacation_planner\\src\\itineraries\\data\\The-Best-100-Travel-Tips-and-Hacks-by-Jessica-Ufuoma-1.pdf']
Starting script...
Finished partitioning articles


No languages specified, defaulting to English.
No languages specified, defaulting to English.
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Finished partitioning PDFs

SOURCE: https://teaspoonofadventure.com/75-questions-for-travellers/
CHUNKS: 8

--- Chunk 0 ---
75 Questions Every Traveller Should Answer

ByRiana Ang-Canning November 1, 2018January 23, 2024

Im going to keep things short and sweet because we have a lot of questions to get to! Read on for 75 questions every traveller should answer. Ive left my answers below but would love to hear yours!

Whats your favourite place so far?

This is one of the toughest questions every traveller should answer. How do you pick just one? I always say that London is my favourite city in the world but I also love Pender Harbour, our annual family vacation spot.

If you could swim with dolphins or go shark diving, which would you pick?

Ive actually done both! I did a dolphin encounter at Atlantis in the Bahamas and went shark cave diving in Durban, South Africa. But both were in enclosed spaces with animals in partial-captivity. If I get the chance to do it in the wild, Ill pick 

In [2]:
# Embed the chunks using a sentence transformer model


model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\Johnson\Desktop\vacation_planner\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Johnson\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2355.59it/s]


In [3]:
# Combine all chunks (articles and documents) 

all_chunks = []

for source, chunks in chunked_articles.items():
    for chunk in chunks:
        all_chunks.append(chunk)

for source, chunks in chunked_documents.items():
    for chunk in chunks:
        all_chunks.append(chunk)

print(f"Total chunks: {len(all_chunks)}")

Total chunks: 43


In [9]:
# Get the text

texts = [chunk.text for chunk in all_chunks]

print(f"Number of texts: {len(texts)}")
print(texts[:10]) 

Number of texts: 43
['75 Questions Every Traveller Should Answer\n\nByRiana Ang-Canning November 1, 2018January 23, 2024\n\nIm going to keep things short and sweet because we have a lot of questions to get to! Read on for 75 questions every traveller should answer. Ive left my answers below but would love to hear yours!\n\nWhats your favourite place so far?\n\nThis is one of the toughest questions every traveller should answer. How do you pick just one? I always say that London is my favourite city in the world but I also love Pender Harbour, our annual family vacation spot.\n\nIf you could swim with dolphins or go shark diving, which would you pick?\n\nIve actually done both! I did a dolphin encounter at Atlantis in the Bahamas and went shark cave diving in Durban, South Africa. But both were in enclosed spaces with animals in partial-captivity. If I get the chance to do it in the wild, Ill pick the dolphins!\n\nWhat place is top of your bucket list?\n\nI hate to say it, but probably 

In [ ]:
# Generating embeddings


embeddings = model.encode(
    texts,
    show_progress_bar=True
)

print(embeddings.shape)

Batches: 100%|██████████| 2/2 [00:01<00:00,  1.32it/s]

(43, 384)


In [11]:
# save the metadata and embeddings to a file

embedded_documents = []

for chunk, embedding in zip(all_chunks, embeddings):
    embedded_documents.append({
        "text": chunk.text,
        "embedding": embedding,
        "metadata": chunk.metadata.to_dict(),
    })

In [14]:
embedded_documents[0]

{'text': '75 Questions Every Traveller Should Answer\n\nByRiana Ang-Canning November 1, 2018January 23, 2024\n\nIm going to keep things short and sweet because we have a lot of questions to get to! Read on for 75 questions every traveller should answer. Ive left my answers below but would love to hear yours!\n\nWhats your favourite place so far?\n\nThis is one of the toughest questions every traveller should answer. How do you pick just one? I always say that London is my favourite city in the world but I also love Pender Harbour, our annual family vacation spot.\n\nIf you could swim with dolphins or go shark diving, which would you pick?\n\nIve actually done both! I did a dolphin encounter at Atlantis in the Bahamas and went shark cave diving in Durban, South Africa. But both were in enclosed spaces with animals in partial-captivity. If I get the chance to do it in the wild, Ill pick the dolphins!\n\nWhat place is top of your bucket list?\n\nI hate to say it, but probably my phone or 